# Project 10 — BROKEN notebook (debugging exercise)

This notebook contains **seeded bugs** that make the model pool the family intercepts too aggressively and sample badly. Run it, read the diagnostics and the shrinkage plot, then fix it. Answer key: `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()
y, x, group, G = data['y'], data['x'], data['group'], data['G']
rates = np.array([y[group==g].mean() for g in range(G)])

### BUGS — centered parameterization AND an over-tight tau prior.

BUG 1: `tau ~ HalfNormal(0.05)` is far too tight — it forces every family intercept toward a single value (over-pooling). BUG 2: the **centered** form re-introduces the funnel, worst exactly where the tight prior pushes $\tau$ (near 0).

In [ ]:
with pm.Model(coords={'group': np.arange(G)}) as model:
    mu = pm.Normal('mu', 0.0, 1.5)
    # BUG 1: over-tight tau prior -> excessive pooling
    tau = pm.HalfNormal('tau', 0.05)
    beta = pm.Normal('beta', 0.0, 1.5)
    # BUG 2: centered parameterization -> funnel
    alpha = pm.Normal('alpha', mu=mu, sigma=tau, dims='group')
    eta = alpha[group] + beta * x
    pm.Bernoulli('y', logit_p=eta, observed=y)
    idata = pm.sample(draws=800, tune=1000, chains=4, target_accept=0.9,
                      random_seed=RNG, progressbar=False)

### Symptom — divergences and a near-zero tau.

In [ ]:
print('divergences:', int(idata.sample_stats['diverging'].sum()))
print(az.summary(idata, var_names=['mu','tau','beta']))

### Diagnostic 1 — energy plot (funnel fingerprint).

In [ ]:
az.plot_energy(idata); plt.tight_layout()

### Diagnostic 2 — the shrinkage pathology.

Plot the recovered family intercepts. Under the over-tight prior they **collapse onto $\hat\mu$** — the model has erased real family differences. Compare to the clean notebook's spread of intercepts.

In [ ]:
alpha_post = idata.posterior['alpha'].mean(dim=('chain','draw')).values
fig, ax = plt.subplots(figsize=(6,3.5))
ax.scatter(range(G), alpha_post, color='#C44E52', label='posterior alpha_g')
ax.axhline(float(idata.posterior['mu'].mean()), color='k', ls='--',
           label='mu_hat')
ax.set(xlabel='family', ylabel='intercept (log-odds)',
       title='Over-pooling: intercepts collapse to the grand mean')
ax.legend(); plt.tight_layout()
print('intercept spread (max-min):', float(alpha_post.max()-alpha_post.min()))

### The fix — non-centered + a sensible tau prior.

Use `tau ~ HalfNormal(1)` (lets the data express real spread) and the non-centered $\alpha_g = \mu + \tau z_g$. Divergences drop to ~0 and the family intercepts regain their genuine spread.

In [ ]:
with pm.Model(coords={'group': np.arange(G)}) as fixed:
    mu = pm.Normal('mu', 0.0, 1.5)
    tau = pm.HalfNormal('tau', 1.0)
    beta = pm.Normal('beta', 0.0, 1.5)
    z = pm.Normal('z', 0.0, 1.0, dims='group')
    alpha = pm.Deterministic('alpha', mu + tau * z, dims='group')
    pm.Bernoulli('y', logit_p=alpha[group] + beta * x, observed=y)
    idata_fixed = pm.sample(draws=800, tune=1000, chains=4,
                            target_accept=0.95, random_seed=RNG,
                            progressbar=False)
a2 = idata_fixed.posterior['alpha'].mean(dim=('chain','draw')).values
print('divergences after fix:', int(idata_fixed.sample_stats['diverging'].sum()))
print('intercept spread after fix:', float(a2.max()-a2.min()))